Welcome to our interactive notebook presenting the results of our paper !

The first cell below reproduces the correlations from the paper (xx Table 4-5, also A2-A3 xx).

The next one generates plots to visualize the differences between global correlation and per-organism correlation.

In [ ]:
from Bio import SeqIO
import pandas as pd
import numpy as np
from sklearn.metrics import root_mean_squared_error as rmse
from scipy.stats import pearsonr

# choose models to include

models = [
    'PRIME',
    'TemStaPro',
    'PPTStab',
    'DeepSTABp',
    'TemBERTure',
    'ThermoFormer',
    'TmProt',
    'FINE_650M_FULL_MELTOME',
    'FINE_650M_FLIP',
    'FINE_650M_NO_INT',
    'FINE_3B_NO_INT'
]

# Choose TPP test sets 
groups_tpp = [
    '1_TEST',
    '3_REVERSED',
    '4_SHUFFLED',
    '2_FIRST_100',
]

# Choose DSC test sets
groups_dsc = [
    'scoop',
    'nanomelt',
    'csp',
    'petases',
]

all_groups = {
    'TPP': groups_tpp,
    'DSC': groups_dsc
}


# Make dict for accessing protein information easily, see cell below for example
big_tpp_file = SeqIO.parse('./data/TPP/tpp.fasta', "fasta")
big_fasta_labels = dict()

for rec in big_tpp_file:
    d = {_n.split('=')[0]: _n.split('=')[-1] for _n in rec.name.split(';')}
    d['tm'] = float(d['tm'])
    d['seq'] = str(rec._seq)  # remove str(..) if you want access to SeqIO.Seq methods
    big_fasta_labels[d['protein_id']] = d

dsc_labels = dict()

for dsc_file in groups_dsc:
    data = SeqIO.parse(f'./data/DSC/{dsc_file}.fasta', "fasta")
    dsc_labels[dsc_file] = dict()
    for rec in data:
        d = {_n.split('=')[0]: _n.split('=')[-1] for _n in rec.name.split(';')}
        d['tm'] = float(d['tm'])
        d['seq'] = str(rec._seq)
        dsc_labels[dsc_file][d['protein_id']] = d

all_labels = {
    'TPP': {g: big_fasta_labels for g in groups_tpp},
    'DSC': dsc_labels
}

In [2]:
# Example: access TPP entry

big_fasta_labels['A0A061ACF5_fbl-1']

{'protein_id': 'A0A061ACF5_fbl-1',
 'up_id': 'A0A061ACF5',
 'mapped_id': 'A0A061ACF5',
 'mapped_parcID': 'NaN',
 'sp': 'C.elegans',
 'set': 'Meltome',
 'split': 'train',
 'tm': 49.459,
 'seq': 'MRICFLLLAFLVAETFANELTRCCAGGTRHFKNSNTCSSIKSEGTSMTCQRAASICCLRSLLDNACDSGTDIAKEEESCPSNINILGGGLKKECCDCCLLAKDLLNRNEPCVAPVGFSAGCLRSFNKCCNGDIEITHASEIITGRPLNDPHVLHLGDRCASSHCEHLCHDRGGEKVECSCRSGFDLAPDGMACVDIDECATLMDDCLESQRCLNTPGSFKCIRTLSCGTGYAMDSETERCRDVDECNLGSHDCGPLYQCRNTQGSYRCDAKKCGDGELQNPMTGECTSITCPNGYYPKNGMCNDIDECVTGHNCGAGEECVNTPGSFRCQQKGNLCAHGYEVNGATGFCEDVNECTTGIAACEQKCVNIPGSYQCICDRGFALGPDGTKCEDIDECSIWAGSGNDLCMGGCINTKGSYLCQCPPGYKIQPDGRTCVDVDECAMGECAGSDKVCVNTLGSFKCHSIDCPTNYIHDSLNKNQIADGYSCIKVCSTEDTECLGNHTREVLYQFRAVPSLKTIISPIEVSRIVTHMGVPFSVDYNLDYVGQRHFRIVQERNIGIVQLVKPISGPTVETIKVNIHTKSRTGVILAFNEAIIEISVSKYPF'}

In [3]:
# Example: access DSC data

dsc_labels['petases']['ACV09511']

{'protein_id': 'ACV09511',
 'tm': 42.4,
 'seq': 'ADTPYERGPNPSESLIEQRRGNYDIAQRSISRFSSDGFRNGTIYYPTDTSDGKFGVIAISPGYTAGESSIAWLGSRIASFGFVVVTINTTTRYDQPRQRSTQLLAALDHAMNDSVVGPLIDPERQAVMGHSMGGGGALQAAESRPEIDAVVALTPWNLKKNWDGVDAATLIIGAERDTVASVRTHSIPFYESLTQAEQRGYLELRGASHFAPNVSNTTIAKYSIAWMKRYLDNDDRYDQFLNPGPAVGYASGVSDYRLQ'}

# Global results (not by organism)

In [ ]:
import warnings

warnings.filterwarnings('ignore')  # For TemStaPro on 'nanomelt' because array is constant (correlation is NaN)

log_NA = False  # Only for TmProt, 25 sequences do not pass conditions proper to the predictor (sequence size, 'X' amino acids, ...)

# 2_FIRST_100 is Truncated
# TmProt model has some constraint (among others, protein sequence size), so only for that model some entries (max 10-20 per organism) are NaN

for mod in models:
    print(mod)
    for group in all_groups:
        names = all_groups[group]
        for name in names:
            result = pd.read_csv(f'./results/{mod}/{group}/{name}.csv') # fetch result file
            if log_NA:
                NAs = result[result['pred'].isna()] # check for NAs (only TmProt)
                print(f'\tWarning: {len(NAs)} NaN in {mod} ({group}): {NAs['name'].tolist()}')
    
            result = result.dropna() # only for TmProt
            names_here = result['name']

            predictions = result['pred']
            exp_tms = [all_labels[group][name][n]['tm'] for n in names_here]
            
            corr, p = pearsonr(predictions, exp_tms)
            n_rmse = rmse(predictions, exp_tms) / np.std(exp_tms)
            print(f"\t{name:15s}: Pearson r = {str(round(corr, 2)):4s} / nRMSE = {str(round(n_rmse, 2)):5s}")
        print()

PRIME
	1_TEST         : Pearson r = 0.84 / nRMSE = 1.62 
	3_REVERSED     : Pearson r = 0.58 / nRMSE = 2.18 
	4_SHUFFLED     : Pearson r = 0.36 / nRMSE = 2.43 
	2_FIRST_100    : Pearson r = 0.73 / nRMSE = 1.82 

	scoop          : Pearson r = 0.61 / nRMSE = 1.96 
	nanomelt       : Pearson r = 0.1  / nRMSE = 3.11 
	csp            : Pearson r = 0.54 / nRMSE = 3.39 
	petases        : Pearson r = 0.18 / nRMSE = 2.95 

TemStaPro
	1_TEST         : Pearson r = 0.68 / nRMSE = 1.34 
	3_REVERSED     : Pearson r = 0.48 / nRMSE = 1.54 
	4_SHUFFLED     : Pearson r = 0.3  / nRMSE = 1.63 
	2_FIRST_100    : Pearson r = 0.39 / nRMSE = 1.42 

	scoop          : Pearson r = 0.61 / nRMSE = 1.68 
	nanomelt       : Pearson r = nan  / nRMSE = 3.63 
	csp            : Pearson r = 0.78 / nRMSE = 1.94 
	petases        : Pearson r = 0.1  / nRMSE = 2.59 

PPTStab
	1_TEST         : Pearson r = 0.77 / nRMSE = 0.67 
	3_REVERSED     : Pearson r = 0.65 / nRMSE = 0.78 
	4_SHUFFLED     : Pearson r = 0.58 / nRMSE = 0.87 
	2_

# By organism

In [6]:
import warnings

# To check all orgas separatly

warnings.filterwarnings('ignore')  # For TemStaPro on 'nanomelt' because array is constant (correlation is NaN)


all_test_orgas = {'A.thaliana',
 'E.coli',
 'Geobacillusthermoleovorans',
 'T.thermophilus',
 'Toxoplasmagondii',
 'Trypanosomacruzi'}

test_groups = {
    'S_int': ['T.thermophilus', 'E.coli', 'A.thaliana'],
    'S_ext': ['Geobacillusthermoleovorans', 'Trypanosomacruzi', 'Toxoplasmagondii']
}

names_by_orga = {
    o: [n for n in big_fasta_labels if big_fasta_labels[n]['sp'] == o] for o in all_test_orgas
}

show_each_orga = False

for mod in models:
    print(mod)
    result = pd.read_csv(f'./results/{mod}/TPP/1_TEST.csv').dropna()
    result_d = result.set_index('name').to_dict()['pred']
    for group in test_groups:
        orgas = test_groups[group]

        all_group_preds = []
        all_group_exp = []

        mean_corr = 0
        mean_nRMSE = 0
        for orga in orgas:
            names_here = names_by_orga[orga]
            names_here = [n for n in names_here if n in result_d] # just for TmProt, has nome NaN because some constraint on protein sequence (like len,...) loses max 10-20 proteins per orga

            predictions = [result_d[n] for n in names_here]
            exp_tms = [all_labels['TPP']['1_TEST'][n]['tm'] for n in names_here]

            all_group_preds += predictions
            all_group_exp += exp_tms
            
            corr, p = pearsonr(predictions, exp_tms)
            n_rmse = rmse(predictions, exp_tms) / np.std(exp_tms)

            mean_corr += corr
            mean_nRMSE += n_rmse
            if show_each_orga:
                print(f"{orga:20s}: Pearson r = {str(round(corr, 2)):4s} / nRMSE = {str(round(n_rmse, 2)):4s}")
        mean_corr /= len(orgas)
        mean_nRMSE /= len(orgas)
        global_corr, _ = pearsonr(all_group_exp, all_group_preds)
        global_nRMSE = rmse(all_group_preds, all_group_exp) / np.std(all_group_exp)

        print(f"{group:5s} (mean): Pearson r = {str(round(mean_corr, 2)):4s} / nRMSE = {str(round(mean_nRMSE, 2)):4s} \t (global): Pearson r = {str(round(global_corr, 2)):4s} / nRMSE = {str(round(global_nRMSE, 2)):4s}")
        print()
    print()

PRIME
S_int (mean): Pearson r = 0.02 / nRMSE = 3.19 	 (global): Pearson r = 0.91 / nRMSE = 1.27

S_ext (mean): Pearson r = 0.01 / nRMSE = 4.55 	 (global): Pearson r = 0.75 / nRMSE = 2.38


TemStaPro
S_int (mean): Pearson r = 0.01 / nRMSE = 2.8  	 (global): Pearson r = 0.78 / nRMSE = 1.14

S_ext (mean): Pearson r = 0.01 / nRMSE = 3.45 	 (global): Pearson r = 0.47 / nRMSE = 1.82


PPTStab
S_int (mean): Pearson r = 0.36 / nRMSE = 1.06 	 (global): Pearson r = 0.92 / nRMSE = 0.4 

S_ext (mean): Pearson r = 0.17 / nRMSE = 1.71 	 (global): Pearson r = 0.34 / nRMSE = 1.15


DeepSTABp
S_int (mean): Pearson r = 0.49 / nRMSE = 0.99 	 (global): Pearson r = 0.93 / nRMSE = 0.37

S_ext (mean): Pearson r = 0.16 / nRMSE = 1.91 	 (global): Pearson r = 0.43 / nRMSE = 1.27


TemBERTure
S_int (mean): Pearson r = 0.48 / nRMSE = 1.06 	 (global): Pearson r = 0.93 / nRMSE = 0.39

S_ext (mean): Pearson r = 0.12 / nRMSE = 1.48 	 (global): Pearson r = 0.66 / nRMSE = 0.89


ThermoFormer
S_int (mean): Pearson r = 0

# Delta $T_m$

In [7]:
import warnings

warnings.filterwarnings('ignore')  # For TemStaPro on 'nanomelt' because array is constant (correlation is NaN)

log_NA = False  # Only for TmProt, 25 sequences do not pass conditions proper to the predictor (sequence size, 'X' amino acids, ...)

todo = [
    '3_REVERSED',
    '4_SHUFFLED',
    '2_FIRST_100',
]

for mod in models:
    print(mod)
    bio_frame = pd.read_csv(f'./results/{mod}/TPP/1_TEST.csv').dropna() # fetch result file
    bio_d = bio_frame.set_index('name').to_dict()['pred']
    for name in todo:
        pert_frame = pd.read_csv(f'./results/{mod}/TPP/{name}.csv').dropna() # fetch result file
        pert_d = pert_frame.set_index('name').to_dict()['pred']
        names_here = set(pert_frame['name']).intersection(set(bio_frame['name']))

        pert_tms = [pert_d[n] for n in names_here]
        bio_preds = [bio_d[n] for n in names_here]
        
        corr, p = pearsonr(pert_tms, bio_preds)
        n_rmse = rmse(pert_tms, bio_preds) / np.std(bio_preds)
        delta_tm = np.mean(np.array(bio_preds) - np.array(pert_tms))

        print(f"Bio. prediction vs {name} prediction: Pearson r = {str(round(corr, 2)):4s} / nRMSE = {str(round(n_rmse, 2)):5s}, delta Tm: {str(round(delta_tm, 2)):5s}")
    print()

PRIME
Bio. prediction vs 3_REVERSED prediction: Pearson r = 0.65 / nRMSE = 0.89 , delta Tm: 6.76 
Bio. prediction vs 4_SHUFFLED prediction: Pearson r = 0.4  / nRMSE = 1.14 , delta Tm: 9.86 
Bio. prediction vs 2_FIRST_100 prediction: Pearson r = 0.84 / nRMSE = 0.57 , delta Tm: 2.33 

TemStaPro
Bio. prediction vs 3_REVERSED prediction: Pearson r = 0.54 / nRMSE = 1.22 , delta Tm: -7.19
Bio. prediction vs 4_SHUFFLED prediction: Pearson r = 0.35 / nRMSE = 1.45 , delta Tm: -10.7
Bio. prediction vs 2_FIRST_100 prediction: Pearson r = 0.57 / nRMSE = 1.16 , delta Tm: -9.83

PPTStab
Bio. prediction vs 3_REVERSED prediction: Pearson r = 0.76 / nRMSE = 0.72 , delta Tm: -1.77
Bio. prediction vs 4_SHUFFLED prediction: Pearson r = 0.71 / nRMSE = 0.72 , delta Tm: 1.16 
Bio. prediction vs 2_FIRST_100 prediction: Pearson r = 0.8  / nRMSE = 0.69 , delta Tm: -2.88

DeepSTABp
Bio. prediction vs 3_REVERSED prediction: Pearson r = 0.71 / nRMSE = 0.81 , delta Tm: 3.08 
Bio. prediction vs 4_SHUFFLED prediction